# 01 V2 - Recovered real dataset audit and frozen subject split

Uses the same Windows CLI as production training. No fixed person, clip, or frame counts. No Google Drive requirement. Input recordings are read-only. Configure POSEFORGE_DATA and optionally POSEFORGE_RUN / POSEFORGE_VAL_SUBJECTS / POSEFORGE_TEST_SUBJECTS. Existing audit evidence is reused; use a new --reports directory to audit a changed dataset. Reports contain no raw video images.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path(os.environ['POSEFORGE_ROOT']).expanduser().resolve() if os.environ.get('POSEFORGE_ROOT') else next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p/'shared').is_dir() and (p/'scripts').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get('POSEFORGE_DATA', ROOT/'artifacts/dataset_working/Real_Posture_Dataset'))
REPORTS = ROOT/'artifacts/reports'
RUN_VALUE = os.environ.get('POSEFORGE_RUN')
RUN = Path(RUN_VALUE).resolve() if RUN_VALUE else None
SEED = 302
DEVICE = os.environ.get('POSEFORGE_DEVICE', 'cuda:0')
# Optional explicit people; None uses a seeded, quality-eligible subject selection.
VAL_PEOPLE = os.environ.get('POSEFORGE_VAL_SUBJECTS', '').split() or None
TEST_PEOPLE = os.environ.get('POSEFORGE_TEST_SUBJECTS', '').split() or None

def stage(name, *extra):
    command = [sys.executable, str(ROOT/'scripts/train_poseforge.py'), '--stage', name,
               '--data', str(DATA), '--reports', str(REPORTS), '--seed', str(SEED)]
    if name not in ('audit', 'prepare'):
        assert RUN is not None, 'Set POSEFORGE_RUN to a prepared real_vNNN directory'
        command += ['--run', str(RUN), '--device', DEVICE, '--resume']
    subprocess.run(command + list(extra), cwd=ROOT, check=True)

def read(path): return json.loads(Path(path).read_text(encoding='utf-8'))
print('Python:',sys.executable,'\nRepository:',ROOT,'\nDataset:',DATA)


In [ ]:
if not (REPORTS/'training_preflight.json').exists():
    stage('audit')
preflight = read(REPORTS/'training_preflight.json')
display(pd.Series({k:preflight[k] for k in ['subject_count','clips','source_takes','frames','pose_frames','eligible_frames','recovery_recognized','no_selected_frames','uncertain_selected_frames']}))
assert not preflight['errors']
print('Dataset ID:',preflight['dataset_id'])

In [ ]:
catalog = pd.read_csv(REPORTS/'clip_quality.csv')
coverage = pd.read_csv(REPORTS/'subject_coverage.csv')
display(coverage.pivot(index='person_id',columns='label',values='eligible'))
display(catalog[catalog.review_required][['clip_id','pose17_frames','usable_frames','uncertain_selected_frames']])

In [ ]:
if RUN is None:
    extra=[]
    if VAL_PEOPLE: extra += ['--validation-subjects', *VAL_PEOPLE]
    if TEST_PEOPLE: extra += ['--test-subjects', *TEST_PEOPLE]
    stage('prepare', *extra)
    RUN = max((ROOT/'artifacts/training').glob('real_v*/split_manifest.json'),key=lambda p:p.stat().st_mtime_ns).parent
split_info=read(RUN/'split_manifest.json')
split_df=pd.read_csv(RUN/'split_manifest.csv')
assert split_info['dataset_id']==preflight['dataset_id']
assert split_df.groupby('person_id').split.nunique().eq(1).all()
assert split_df.groupby('source_group_id').split.nunique().eq(1).all()
print('Frozen run:',RUN)
print('Set POSEFORGE_RUN to this directory for notebooks 02 and 03.')
display(split_df.groupby(['split','label']).agg(clips=('clip_id','count'),subjects=('person_id','nunique')))

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4))
catalog.groupby('label').frames.sum().plot.bar(ax=axes[0],title='Actual source frames by label')
coverage.pivot(index='person_id',columns='label',values='eligible').plot.bar(stacked=True,ax=axes[1],title='Eligible frames per subject')
plt.tight_layout();plt.show()

The training snapshot records actual source lineage and hashes. It is an automated quality export, not a claim of manual posture approval. Missing joints do not remove a clip; uncertain frames are excluded from supervision. No overlapping segments are generated.